In [1]:
from sklearn.neighbors import KDTree

from darth.darth_functions import *
from visualisation import *

with open("darth/config.json", "r") as file:
    config = json.load(file)

/home/hossein/CICCADA/ciccada/lib/python3.12/site-packages/paramiko/pkey.py:82: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from cryptography.hazmat.primitives.ciphers.algorithms in 48.0.0.
  "cipher": algorithms.TripleDES,
/home/hossein/CICCADA/ciccada/lib/python3.12/site-packages/paramiko/transport.py:253: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from cryptography.hazmat.primitives.ciphers.algorithms in 48.0.0.
  "class": algorithms.TripleDES,


In [6]:
# bom_data_ant_v
# bom_data_nsw_v
# bom_data_nt_v
# bom_data_qld_v
# bom_data_sa_v
# bom_data_tas_v
# bom_data_vic_v
# bom_data_wa_v
# bom_station_details_v
tablename = "bom_station_details_v"

In [15]:
site_locations = pd.read_csv("tests/darth/site_locations.csv")

In [ ]:
import json
import os
import urllib.parse

import pandas as pd
import psycopg2
import yaml
from sqlalchemy import create_engine
from sshtunnel import SSHTunnelForwarder

hi


In [7]:
with SSHTunnelForwarder(
    ssh_address_or_host=(config["hostserver"], 22),
    ssh_username=config["ssh_username"],
    ssh_pkey=config["ssh_private_key_path"],
    ssh_private_key_password=config["ssh_private_key_password"],
    host_pkey_directories=[],
    remote_bind_address=(config["remote_host"], config["remote_port"]),
) as tunnel:
    # tunnel.start()
    print("hi")
    # local_port = str(tunnel.local_bind_port)

    # # Needed to handle special characters in password i.e. "@"
    # darth_password = urllib.parse.quote_plus(config["darth_password"])

    # engine_str = 'postgresql+psycopg2://{user}:{password}@{host}:{port}/{db}'.format(
    #     user=config["darth_username"],
    #     password=darth_password,
    #     host='localhost',
    #     port=local_port,
    #     db=config["databasename"]
    # )

    # engine = create_engine(engine_str)
    # con = engine.raw_connection()

2026-01-11 23:24:38,229| ERROR   | Could not connect to gateway 54.253.60.172:22 : Unable to connect to 54.253.60.172: [Errno 110] Connection timed out


BaseSSHTunnelForwarderError: Could not establish session to SSH gateway

In [2]:
query = f"""select * from bom_station_details_v 
where station_number='067119' limit 1"""
get_darth_data(config, query)

AttributeError: 'NoneType' object has no attribute 'start'

In [13]:
query = f"""select station_number, datetime, air_temperature_in_degrees_c as temp
    from bom_data_nsw_v
    where station_number = '067119' and year=2024 and extract(hour from datetime) in (10, 11, 12, 13, 14)
    group by station_number, date_trunc('day', datetime)
    order by day asc
    """
get_darth_data(config, query)

,station_number,day,temp
0,67119,2024-01-01,22.3
1,67119,2024-01-02,28.0
2,67119,2024-01-03,30.5
3,67119,2024-01-04,27.6
4,67119,2024-01-05,24.0
...,...,...,...
361,67119,2024-12-27,36.4
362,67119,2024-12-28,27.2
363,67119,2024-12-29,29.9
364,67119,2024-12-30,27.6


In [17]:
# 146.70	-19.395
67119
query = f"""select station_number, longitude, latitude from bom_station_details_v"""
station_locations = get_darth_data(config, query)
station_locations[:2]

,station_number,longitude,latitude
0,001006,128.1503,-15.5100
1,001007,126.1485,-13.7542


In [18]:
station_coords = station_locations[["latitude", "longitude"]].to_numpy()
site_coords = site_locations[["latitude", "longitude"]].to_numpy()
kdtree = KDTree(station_coords, metric="euclidean")
distances, indices = kdtree.query(site_coords, k=1)  # k=1 → nearest
nearest_indices = indices.flatten()
nearest_distances = distances.flatten()
site_locations[["n_long", "n_lat", "station_number"]] = station_locations.iloc[
    nearest_indices
][["longitude", "latitude", "station_number"]].values
site_locations["distance_km"] = (
    nearest_distances * 111
)  # Rough conversion factor for degrees to kilometers

In [28]:
site_locations.query("site_id == 1299741610")

,site_id,state,longitude,latitude,n_long,n_lat,station_number,distance_km
2955,1299741610,NSW,150.9,-33.705,150.8567,-33.8510,067119,16.903697


In [27]:
",".join(site_locations["station_number"].unique())

'039089,033119,040861,066062,058077,080091,066051,061055,040764,055325,032040,065070,069147,140009,070014,069138,033317,060168,040913,067119,033329,040405,058009,087031,067113,041097,040922,031222,087168,140008,032195,066194,066161,140007,066193,074272,070330,040211,040925,091344,040958,066037,033327,068257,086220,061412,061078,023013,033247,033002,040842,040988,040717,063303,040004,067117,023123,066200,085280,094191,072150,074148,094029,068192,081125,086372,008051,039123,061379,089002,40913,066195,040927,062101,058214,009519,076031,072160,40284,040126,041525,023000,070351,009214,059007,033255,086388,040082,009091,040223,088162,033328,039128,061425,066137,082138,041529,040284,014932,031011,067033,068263,009887,068242,058216,066214,023090,085313,086071,039083,066212,040983,009240,61250,092114,081123,070339,023849,023842,049136,095003,067105,077094,040282,003032,066043,023046,023109,023083,033045,088051,086068,024584,009172,073151,023373,040908,055202,085296,091126,018192,086104,014518,0

In [31]:
# {','.join(site_locations['station_number'].unique())}
df_list = []
state = ["ant", "nsw", "vic", "tas", "qld", "sa", "nt", "wa"]
offset = [10, 10, 10, 10, 10, 9.5, 9.5, 8]
# state  = ['nsw']
# offset = [ 10 ]
for s, o in zip(state, offset):
    query = f"""select station_number, date_trunc('day', datetime)  as day, max(air_temperature_in_degrees_c) as temp
    from bom_data_{s}_v
    where station_number in ({",".join(site_locations["station_number"].unique())}) and year=2024 and extract(hour from datetime) in (10, 11, 12, 13, 14)
    group by station_number, date_trunc('day', datetime)
    order by day asc
    """
    df = get_darth_data(config, query)
    # df['time'] = pd.to_datetime(df['time']).dt.tz_localize(pytz.FixedOffset(60*o))
    df_list.append(df)
df = pd.concat(df_list, ignore_index=True)

In [32]:
df[df["station_number"] == 67119]

,station_number,day,temp
69,67119,2024-01-01,22.3
177,67119,2024-01-02,28.0
286,67119,2024-01-03,30.5
395,67119,2024-01-04,27.6
504,67119,2024-01-05,24.0
...,...,...,...
39252,67119,2024-12-27,36.4
39361,67119,2024-12-28,27.2
39469,67119,2024-12-29,29.9
39577,67119,2024-12-30,27.6


In [33]:
df.to_parquet("tests/darth/station_temp_2024.parquet", index=False)

In [42]:
site_locations[["site_id", "station_number"]].to_parquet(
    "tests/darth/site_station_mapping.parquet", index=False
)

In [4]:
start_time = "2024-01-01 00:00:00+11:00"  # In sydney local time
end_time = "2024-02-01 00:00:00+11:00"  # In sydney local time

num_ticks = 24 * 2 + 1
# save_as = 'Figures/EDP_voltwatt_12Nov.jpeg'
save_as = ""
x_label = "time"
y_labels = ["temp (^oC)"]

plt_config = {"temp": [0, 0, "-", None, None]}

color_nights = False
# color_by = 'group'
color_by = "attribute"
ax_digit = "1.1f"
a = my_plot4(
    start_time,
    end_time,
    df,
    plt_config=plt_config,
    ax_digit=ax_digit,
    group_attr="station_number",
    time_attr="time",
    color_nights=color_nights,
    cmap="viridis",
    figsize=[12 / 2.54, 1.5],
    same_scale=1,
    fontsize=7,
    fontname="Times New Roman",
    plot_total=False,
    plot_total_func=["sum", [lambda x: max(x), "max"]],
    num_ticks=num_ticks,
    num_yticks=10,
    dpi=300,
    x_format="%H:%M",
    legend_loc=["upper left"],
    x_label=x_label,
    y_labels=y_labels,
    color_by=color_by,
    plot_period=np.timedelta64(1, "D"),
    save_as=save_as,
    rotation=60,
    step=0,
    gridwidth=[0.2, 0.2],
    legend_join="-",
    title="",
    legend_i=0,
    legend_j=None,
    title_i=0,
    only1title=0,
)
a.do()

NameError: name 'df' is not defined

In [6]:
os.path.exists(config["ssh_private_key_path"])

True

In [12]:
edp_path = "/home/hossein/CICCADA/tests/4) Data/EDP SA 2023 Data"
meta_data1 = pd.read_csv(edp_path + "/edp_sites_metadata_sa_postcode.csv")
meta_data2 = pd.read_csv(edp_path + "/edp_sites_metadata59239829.csv")
meta_data2 = meta_data2[meta_data2["state"] == "SA"]
SA_site_ids = meta_data2["edp_site_id"].unique()

In [9]:
query = f"select max(datetime), min(datetime)  from {tablename} "
data = get_darth_data(config, query)
data

,max,min
0,2022-11-30 23:55:00,2022-11-01


In [ ]:
query = f"select *  from {tablename} limit 1"
data = get_darth_data(config, query)
data

,edp_site_id,unix_time,datetime,edp_device_and_circuit,circuit_label,edp_circuit_label,real_energy,real_energy_negative,real_energy_positive,reactive_energy,current_avg,current_min,current_max,voltage_avg,voltage_min,voltage_max
0,S0002,1667260800,2022-11-01,A1,pv_site_net,pv_site_net_A1,-0.0172,0.0172,0.0,2.8911,0.1415,None,None,246.05,None,None


In [13]:
query = f"select count(distinct edp_site_id)  from {tablename} where edp_site_id in {tuple(SA_site_ids)}"
data = get_darth_data(config, query)
data

,count
0,141


In [16]:
query = f"select *  from {tablename} \
    where edp_site_id in {tuple(SA_site_ids)} and \
circuit_label = 'pv_site_net'"
data = get_darth_data(config, query)
data

,edp_site_id,unix_time,datetime,edp_device_and_circuit,circuit_label,edp_circuit_label,real_energy,real_energy_negative,real_energy_positive,reactive_energy,current_avg,current_min,current_max,voltage_avg,voltage_min,voltage_max
0,S0031,1667260800,2022-11-01 00:00:00,B1,pv_site_net,pv_site_net_B1,-0.0833,0.0833,0.0,5.2958,0.2640,NaN,NaN,241.90,NaN,NaN
1,S0031,1667260800,2022-11-01 00:00:00,B2,pv_site_net,pv_site_net_B2,-0.0594,0.0594,0.0,1.8292,0.0890,NaN,NaN,243.50,NaN,NaN
2,S0031,1667261100,2022-11-01 00:05:00,B1,pv_site_net,pv_site_net_B1,-0.0856,0.0856,0.0,5.2750,0.2635,NaN,NaN,241.35,NaN,NaN
3,S0031,1667261100,2022-11-01 00:05:00,B2,pv_site_net,pv_site_net_B2,-0.0667,0.0667,0.0,1.8156,0.0885,NaN,NaN,243.20,NaN,NaN
4,S0031,1667261400,2022-11-01 00:10:00,B1,pv_site_net,pv_site_net_B1,-0.0853,0.0853,0.0,5.2308,0.2625,NaN,NaN,240.25,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1616803,W0325,1669812600,2022-11-30 23:50:00,A5,pv_site_net,pv_site_net_A5,-0.0500,0.0500,0.0,3.2236,NaN,0.159,0.161,NaN,242.0,243.0
1616804,W0325,1669812600,2022-11-30 23:50:00,A6,pv_site_net,pv_site_net_A6,-0.0678,0.0678,0.0,3.1950,NaN,0.156,0.159,NaN,243.1,243.9
1616805,W0325,1669812900,2022-11-30 23:55:00,A4,pv_site_net,pv_site_net_A4,-0.1986,0.1986,0.0,3.4153,NaN,0.169,0.172,NaN,242.0,242.6
1616806,W0325,1669812900,2022-11-30 23:55:00,A5,pv_site_net,pv_site_net_A5,-0.0500,0.0500,0.0,3.2272,NaN,0.158,0.161,NaN,242.2,242.9


In [22]:
data.to_csv("/home/hossein/CICCADA/tests/edp_data_2022_11_v.csv", index=False)